### import kaggle ds using kaggle hub

In [2]:
import kagglehub
uciml_adult_census_income_path = kagglehub.dataset_download('uciml/adult-census-income')

print('Data source import complete.')
print(uciml_adult_census_income_path)

100%|██████████| 450k/450k [00:00<00:00, 635kB/s]

Extracting files...
Data source import complete.
/root/.cache/kagglehub/datasets/uciml/adult-census-income/versions/3


### download the libs

In [3]:
!pip install pyspark -q

In [4]:
import pandas as pd
import numpy as np

from pyspark.sql import SparkSession
from pyspark.sql.functions import col

### create spark session

In [5]:
spark = (
    SparkSession.builder
    .appName("Adult Census Income Classification")
    .master("local[*]")
    .getOrCreate()
)

print("Spark Version :", spark.version)

Spark Version : 4.0.3


Load the DS

In [6]:
import os

for root, dirs, files in os.walk(uciml_adult_census_income_path):
    for file in files:
        print(os.path.join(root, file))


df = spark.read.csv(
    f"{uciml_adult_census_income_path}/adult.csv",
    header=True,
    inferSchema=True
)

/root/.cache/kagglehub/datasets/uciml/adult-census-income/versions/3/adult.csv


### Some basic explorations

In [7]:
df.printSchema()

root
 |-- age: integer (nullable = true)
 |-- workclass: string (nullable = true)
 |-- fnlwgt: integer (nullable = true)
 |-- education: string (nullable = true)
 |-- education.num: integer (nullable = true)
 |-- marital.status: string (nullable = true)
 |-- occupation: string (nullable = true)
 |-- relationship: string (nullable = true)
 |-- race: string (nullable = true)
 |-- sex: string (nullable = true)
 |-- capital.gain: integer (nullable = true)
 |-- capital.loss: integer (nullable = true)
 |-- hours.per.week: integer (nullable = true)
 |-- native.country: string (nullable = true)
 |-- income: string (nullable = true)



In [8]:
print("Rows :", df.count())
print("Columns :", len(df.columns))
df.show(5, truncate=False)

Rows : 32561
Columns : 15
+---+---------+------+------------+-------------+--------------+-----------------+-------------+-----+------+------------+------------+--------------+--------------+------+
|age|workclass|fnlwgt|education   |education.num|marital.status|occupation       |relationship |race |sex   |capital.gain|capital.loss|hours.per.week|native.country|income|
+---+---------+------+------------+-------------+--------------+-----------------+-------------+-----+------+------------+------------+--------------+--------------+------+
|90 |?        |77053 |HS-grad     |9            |Widowed       |?                |Not-in-family|White|Female|0           |4356        |40            |United-States |<=50K |
|82 |Private  |132870|HS-grad     |9            |Widowed       |Exec-managerial  |Not-in-family|White|Female|0           |4356        |18            |United-States |<=50K |
|66 |?        |186061|Some-college|10           |Widowed       |?                |Unmarried    |Black|Female|

In [9]:
print(df.columns)
print("data types for each colms: ")
print(df.dtypes)

['age', 'workclass', 'fnlwgt', 'education', 'education.num', 'marital.status', 'occupation', 'relationship', 'race', 'sex', 'capital.gain', 'capital.loss', 'hours.per.week', 'native.country', 'income']
data types for each colms: 
[('age', 'int'), ('workclass', 'string'), ('fnlwgt', 'int'), ('education', 'string'), ('education.num', 'int'), ('marital.status', 'string'), ('occupation', 'string'), ('relationship', 'string'), ('race', 'string'), ('sex', 'string'), ('capital.gain', 'int'), ('capital.loss', 'int'), ('hours.per.week', 'int'), ('native.country', 'string'), ('income', 'string')]


### EDA

check if missing values

In [10]:
print("=" * 60)
print("Adult Census Income Dataset")
print("=" * 60)

print(f"Number of Rows    : {df.count()}")
print(f"Number of Columns : {len(df.columns)}")

print("\nColumns:")
for col_name in df.columns:
    print("-", col_name)

Adult Census Income Dataset
Number of Rows    : 32561
Number of Columns : 15

Columns:
- age
- workclass
- fnlwgt
- education
- education.num
- marital.status
- occupation
- relationship
- race
- sex
- capital.gain
- capital.loss
- hours.per.week
- native.country
- income


Here the education.num or martial.status types of colms where confusing the spark as within its conventions the '.' opr causes it to think that this is a nested colms but they are not nested and hence we run a the code below to create a new DF and replace all '.' with '_'

In [11]:
df = df.toDF(*[
    column.replace(".", "_")
    for column in df.columns
])

df.printSchema()

root
 |-- age: integer (nullable = true)
 |-- workclass: string (nullable = true)
 |-- fnlwgt: integer (nullable = true)
 |-- education: string (nullable = true)
 |-- education_num: integer (nullable = true)
 |-- marital_status: string (nullable = true)
 |-- occupation: string (nullable = true)
 |-- relationship: string (nullable = true)
 |-- race: string (nullable = true)
 |-- sex: string (nullable = true)
 |-- capital_gain: integer (nullable = true)
 |-- capital_loss: integer (nullable = true)
 |-- hours_per_week: integer (nullable = true)
 |-- native_country: string (nullable = true)
 |-- income: string (nullable = true)



here this cmd is equivalent to pandas describe()
> here cat colms have null for:

>stddev & mean


In [12]:
df.describe().show()

+-------+------------------+-----------+------------------+------------+-----------------+--------------+----------------+------------+------------------+------+------------------+-----------------+------------------+--------------+------+
|summary|               age|  workclass|            fnlwgt|   education|    education_num|marital_status|      occupation|relationship|              race|   sex|      capital_gain|     capital_loss|    hours_per_week|native_country|income|
+-------+------------------+-----------+------------------+------------+-----------------+--------------+----------------+------------+------------------+------+------------------+-----------------+------------------+--------------+------+
|  count|             32561|      32561|             32561|       32561|            32561|         32561|           32561|       32561|             32561| 32561|             32561|            32561|             32561|         32561| 32561|
|   mean| 38.58164675532078|       NULL|

check for null vals here the ds uses '?' instead of NULL for null vals

In [13]:
from pyspark.sql.functions import col, when, count

df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show(vertical=True)

-RECORD 0-------------
 age            | 0   
 workclass      | 0   
 fnlwgt         | 0   
 education      | 0   
 education_num  | 0   
 marital_status | 0   
 occupation     | 0   
 relationship   | 0   
 race           | 0   
 sex            | 0   
 capital_gain   | 0   
 capital_loss   | 0   
 hours_per_week | 0   
 native_country | 0   
 income         | 0   



In [14]:
from pyspark.sql.functions import trim

string_cols = [
    c for c, t in df.dtypes
    if t == "string"
]


for colm in string_cols:
  cnt = df.filter(
        trim(col(colm)) == "?"
      ).count()
  print(f"{colm:20} : {cnt}")

workclass            : 1836
education            : 0
marital_status       : 0
occupation           : 1843
relationship         : 0
race                 : 0
sex                  : 0
native_country       : 583
income               : 0


In [15]:
df.groupBy("income").count().show()

+------+-----+
|income|count|
+------+-----+
| <=50K|24720|
|  >50K| 7841|
+------+-----+



Inspecting the Num colms

In [16]:
numerical_cols = [
    "age",
    "fnlwgt",
    "education_num",
    "capital_gain",
    "capital_loss",
    "hours_per_week"
]

df.select(numerical_cols).describe().show()

+-------+------------------+------------------+-----------------+------------------+-----------------+------------------+
|summary|               age|            fnlwgt|    education_num|      capital_gain|     capital_loss|    hours_per_week|
+-------+------------------+------------------+-----------------+------------------+-----------------+------------------+
|  count|             32561|             32561|            32561|             32561|            32561|             32561|
|   mean| 38.58164675532078|189778.36651208502| 10.0806793403151|1077.6488437087312|  87.303829734959|40.437455852092995|
| stddev|13.640432553581295|105549.97769702264|2.572720332067391| 7385.292084840311|402.9602186489979|12.347428681731857|
|    min|                17|             12285|                1|                 0|                0|                 1|
|    max|                90|           1484705|               16|             99999|             4356|                99|
+-------+---------------

Cat colms


In [17]:
categorical_cols = [
    "workclass",
    "education",
    "marital_status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "native_country"
]

for column in categorical_cols:
    print(f"\n{column}")
    df.groupBy(column).count().orderBy("count", ascending=False).show(5)


workclass
+----------------+-----+
|       workclass|count|
+----------------+-----+
|         Private|22696|
|Self-emp-not-inc| 2541|
|       Local-gov| 2093|
|               ?| 1836|
|       State-gov| 1298|
+----------------+-----+
only showing top 5 rows

education
+------------+-----+
|   education|count|
+------------+-----+
|     HS-grad|10501|
|Some-college| 7291|
|   Bachelors| 5355|
|     Masters| 1723|
|   Assoc-voc| 1382|
+------------+-----+
only showing top 5 rows

marital_status
+------------------+-----+
|    marital_status|count|
+------------------+-----+
|Married-civ-spouse|14976|
|     Never-married|10683|
|          Divorced| 4443|
|         Separated| 1025|
|           Widowed|  993|
+------------------+-----+
only showing top 5 rows

occupation
+---------------+-----+
|     occupation|count|
+---------------+-----+
| Prof-specialty| 4140|
|   Craft-repair| 4099|
|Exec-managerial| 4066|
|   Adm-clerical| 3770|
|          Sales| 3650|
+---------------+-----+
only 

In [18]:
freq_cols = ["education", "sex", "race", "occupation"]

for c in freq_cols:
  df.groupBy(c) \
    .count() \
    .orderBy("count", ascending=False) \
    .show()

+------------+-----+
|   education|count|
+------------+-----+
|     HS-grad|10501|
|Some-college| 7291|
|   Bachelors| 5355|
|     Masters| 1723|
|   Assoc-voc| 1382|
|        11th| 1175|
|  Assoc-acdm| 1067|
|        10th|  933|
|     7th-8th|  646|
| Prof-school|  576|
|         9th|  514|
|        12th|  433|
|   Doctorate|  413|
|     5th-6th|  333|
|     1st-4th|  168|
|   Preschool|   51|
+------------+-----+

+------+-----+
|   sex|count|
+------+-----+
|  Male|21790|
|Female|10771|
+------+-----+

+------------------+-----+
|              race|count|
+------------------+-----+
|             White|27816|
|             Black| 3124|
|Asian-Pac-Islander| 1039|
|Amer-Indian-Eskimo|  311|
|             Other|  271|
+------------------+-----+

+-----------------+-----+
|       occupation|count|
+-----------------+-----+
|   Prof-specialty| 4140|
|     Craft-repair| 4099|
|  Exec-managerial| 4066|
|     Adm-clerical| 3770|
|            Sales| 3650|
|    Other-service| 3295|
|Machine-o

In [19]:
numerical_cols = [
    "age",
    "education_num",
    "capital_gain",
    "capital_loss",
    "hours_per_week"
]

for column in numerical_cols:
    corr = df.stat.corr(column, "education_num")
    print(f"Correlation({column}, education_num) = {corr:.3f}")

Correlation(age, education_num) = 0.037
Correlation(education_num, education_num) = 1.000
Correlation(capital_gain, education_num) = 0.123
Correlation(capital_loss, education_num) = 0.080
Correlation(hours_per_week, education_num) = 0.148


In [20]:
duplicates = df.count() - df.dropDuplicates().count()

print("Duplicate Records :", duplicates)

Duplicate Records : 24


In [21]:
df.printSchema()

root
 |-- age: integer (nullable = true)
 |-- workclass: string (nullable = true)
 |-- fnlwgt: integer (nullable = true)
 |-- education: string (nullable = true)
 |-- education_num: integer (nullable = true)
 |-- marital_status: string (nullable = true)
 |-- occupation: string (nullable = true)
 |-- relationship: string (nullable = true)
 |-- race: string (nullable = true)
 |-- sex: string (nullable = true)
 |-- capital_gain: integer (nullable = true)
 |-- capital_loss: integer (nullable = true)
 |-- hours_per_week: integer (nullable = true)
 |-- native_country: string (nullable = true)
 |-- income: string (nullable = true)



### Data Cleaning

In [22]:
from pyspark.sql.functions import col, trim, when
null_cnt = df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show(vertical=False)

null_cnt

-RECORD 0-------------
 age            | 0   
 workclass      | 0   
 fnlwgt         | 0   
 education      | 0   
 education_num  | 0   
 marital_status | 0   
 occupation     | 0   
 relationship   | 0   
 race           | 0   
 sex            | 0   
 capital_gain   | 0   
 capital_loss   | 0   
 hours_per_week | 0   
 native_country | 0   
 income         | 0   



In [23]:
string_col = [
    c for c, t in df.dtypes
    if t == "string"
]

for colm in string_col:
  missing = df.filter(trim(col(colm)) == "?").count()
  print(f"{colm:20} : {missing}")

workclass            : 1836
education            : 0
marital_status       : 0
occupation           : 1843
relationship         : 0
race                 : 0
sex                  : 0
native_country       : 583
income               : 0


In [53]:
for colm in string_col:
  df = df.withColumn(
      colm,
      when(trim(col(colm)) == "?", None)
      .otherwise(trim(col(colm))))
df.show(5, trunacate)

+---+----------+------+----------+-------------+--------------+----------+------------+-----+------+------------+------------+--------------+--------------+------+-----+---------------+---------------+--------------------+----------------+------------------+----------+---------+--------------------+-------------+-------------+------------------+--------------+----------------+----------+----------+------------------+----------+---------------+
|age| workclass|fnlwgt| education|education_num|marital_status|occupation|relationship| race|   sex|capital_gain|capital_loss|hours_per_week|native_country|income|label|workclass_index|education_index|marital_status_index|occupation_index|relationship_index|race_index|sex_index|native_country_index|workclass_vec|education_vec|marital_status_vec|occupation_vec|relationship_vec|  race_vec|   sex_vec|native_country_vec|  features|scaled_features|
+---+----------+------+----------+-------------+--------------+----------+------------+-----+------+----

In [25]:
df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in string_col
]).show(vertical=True)

-RECORD 0--------------
 workclass      | 1836 
 education      | 0    
 marital_status | 0    
 occupation     | 1843 
 relationship   | 0    
 race           | 0    
 sex            | 0    
 native_country | 583  
 income         | 0    



In [26]:
before = df.count()

df = df.dropna()

after = df.count()

print(f"Rows before cleaning : {before}")
print(f"Rows after cleaning  : {after}")
print(f"Rows removed         : {before - after}")

Rows before cleaning : 32561
Rows after cleaning  : 30162
Rows removed         : 2399


In [27]:
duplicates = df.count() - df.dropDuplicates().count()

print("Duplicate Records :", duplicates)

Duplicate Records : 23


In [28]:
print("dropping dublicates")
df = df.dropDuplicates()

dropping dublicates


In [29]:
print("Final Dataset Shape")

print("Rows :", df.count())
print("Columns :", len(df.columns))

df.printSchema()

Final Dataset Shape
Rows : 30139
Columns : 15
root
 |-- age: integer (nullable = true)
 |-- workclass: string (nullable = true)
 |-- fnlwgt: integer (nullable = true)
 |-- education: string (nullable = true)
 |-- education_num: integer (nullable = true)
 |-- marital_status: string (nullable = true)
 |-- occupation: string (nullable = true)
 |-- relationship: string (nullable = true)
 |-- race: string (nullable = true)
 |-- sex: string (nullable = true)
 |-- capital_gain: integer (nullable = true)
 |-- capital_loss: integer (nullable = true)
 |-- hours_per_week: integer (nullable = true)
 |-- native_country: string (nullable = true)
 |-- income: string (nullable = true)



Caching the cleaned df so that we dont have to compute them again and again

In [30]:
df = df.cache()
print(f"Rows after cleaning: {df.count()}")

Rows after cleaning: 30139


### Feature engineering


identify the feat colms

In [31]:
label_col = "income"

categorical_cols = [
    "workclass",
    "education",
    "marital_status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "native_country"
]

numerical_cols = [
    "age",
    "fnlwgt",
    "education_num",
    "capital_gain",
    "capital_loss",
    "hours_per_week"
]

In [32]:
print("Categorical Features:")
print(categorical_cols)

print("\nNumerical Features:")
print(numerical_cols)

Categorical Features:
['workclass', 'education', 'marital_status', 'occupation', 'relationship', 'race', 'sex', 'native_country']

Numerical Features:
['age', 'fnlwgt', 'education_num', 'capital_gain', 'capital_loss', 'hours_per_week']


Encode Target Variable by StringIndexer

In [33]:
from pyspark.ml.feature import StringIndexer

label_indexer = StringIndexer(
    inputCol="income",
    outputCol="label"
)

df = label_indexer.fit(df).transform(df)

In [34]:
df.select("income", "label").show(10)

+------+-----+
|income|label|
+------+-----+
|  >50K|  1.0|
| <=50K|  0.0|
| <=50K|  0.0|
| <=50K|  0.0|
| <=50K|  0.0|
|  >50K|  1.0|
| <=50K|  0.0|
| <=50K|  0.0|
|  >50K|  1.0|
|  >50K|  1.0|
+------+-----+
only showing top 10 rows


In [35]:
feature_indexers = [
    StringIndexer(
        inputCol=column,
        outputCol=f"{column}_index",
        handleInvalid="keep"
    )
    for column in categorical_cols
]

for indexer in feature_indexers:
    df = indexer.fit(df).transform(df)

In [36]:
df.select(
    "workclass",
    "workclass_index"
).distinct().show()

+----------------+---------------+
|       workclass|workclass_index|
+----------------+---------------+
|       State-gov|            3.0|
|         Private|            0.0|
|Self-emp-not-inc|            1.0|
|       Local-gov|            2.0|
|     Federal-gov|            5.0|
|    Self-emp-inc|            4.0|
|     Without-pay|            6.0|
+----------------+---------------+



Now applying OHE on the string-indexed colms



as the **ML algos in spark cannot perform on the string** we mapped them to idices 1,2,3...n so that we can sent them to ML models and then we r **performing OHE on this indices** as thery are in nature the Cat_Colms

In [37]:
from pyspark.ml.feature import OneHotEncoder
encoder = OneHotEncoder(
    inputCols=[f"{c}_index" for c in categorical_cols],
    outputCols=[f"{c}_vec" for c in categorical_cols]
)

df = encoder.fit(df).transform(df)

df.printSchema()

root
 |-- age: integer (nullable = true)
 |-- workclass: string (nullable = true)
 |-- fnlwgt: integer (nullable = true)
 |-- education: string (nullable = true)
 |-- education_num: integer (nullable = true)
 |-- marital_status: string (nullable = true)
 |-- occupation: string (nullable = true)
 |-- relationship: string (nullable = true)
 |-- race: string (nullable = true)
 |-- sex: string (nullable = true)
 |-- capital_gain: integer (nullable = true)
 |-- capital_loss: integer (nullable = true)
 |-- hours_per_week: integer (nullable = true)
 |-- native_country: string (nullable = true)
 |-- income: string (nullable = true)
 |-- label: double (nullable = false)
 |-- workclass_index: double (nullable = false)
 |-- education_index: double (nullable = false)
 |-- marital_status_index: double (nullable = false)
 |-- occupation_index: double (nullable = false)
 |-- relationship_index: double (nullable = false)
 |-- race_index: double (nullable = false)
 |-- sex_index: double (nullable = fal

### Vector Assembler

In [38]:
from pyspark.ml.feature import VectorAssembler

assembler_inputs = numerical_cols + [
    f"{c}_vec"
    for c in categorical_cols
]

assembler_inputs

['age',
 'fnlwgt',
 'education_num',
 'capital_gain',
 'capital_loss',
 'hours_per_week',
 'workclass_vec',
 'education_vec',
 'marital_status_vec',
 'occupation_vec',
 'relationship_vec',
 'race_vec',
 'sex_vec',
 'native_country_vec']

In [39]:
assembler = VectorAssembler(
    inputCols=assembler_inputs,
    outputCol="features"
)

df = assembler.transform(df)

In [40]:
df.select("features", "label").show(5, truncate=False)

+---------------------------------------------------------------------------------------------------------+-----+
|features                                                                                                 |label|
+---------------------------------------------------------------------------------------------------------+-----+
|(104,[0,1,2,4,5,10,13,29,39,54,56,62,63],[49.0,158685.0,9.0,2377.0,40.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0])|1.0  |
|(104,[0,1,2,4,5,6,13,29,43,50,56,61,63],[34.0,265807.0,9.0,2051.0,55.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]) |0.0  |
|(104,[0,1,2,4,5,6,14,29,44,50,56,61,63],[35.0,40135.0,10.0,2042.0,40.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]) |0.0  |
|(104,[0,1,2,4,5,6,14,30,39,51,56,62,63],[26.0,58098.0,10.0,1974.0,40.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]) |0.0  |
|(104,[0,1,2,4,5,6,15,30,38,51,56,61,63],[26.0,215384.0,13.0,1974.0,55.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0])|0.0  |
+---------------------------------------------------------------------------------------

### StandardScaler

In [41]:
from pyspark.ml.feature import StandardScaler

scaler = StandardScaler(
    inputCol="features",
    outputCol="scaled_features",
    withMean=True,
    withStd=True
)

scaler_model = scaler.fit(df)

df = scaler_model.transform(df)

In [42]:
df.select(
    "features",
    "scaled_features"
).show(5, truncate=True)

+--------------------+--------------------+
|            features|     scaled_features|
+--------------------+--------------------+
|(104,[0,1,2,4,5,1...|[0.80404671911021...|
|(104,[0,1,2,4,5,6...|[-0.3382511572031...|
|(104,[0,1,2,4,5,6...|[-0.2620979654489...|
|(104,[0,1,2,4,5,6...|[-0.9474766912369...|
|(104,[0,1,2,4,5,6...|[-0.9474766912369...|
+--------------------+--------------------+
only showing top 5 rows


## Final DF

In [54]:
final_df = df.select(
    "scaled_features",
    "label"
)

final_df.show(5, truncate=True)

+--------------------+-----+
|     scaled_features|label|
+--------------------+-----+
|[0.80404671911021...|  1.0|
|[-0.3382511572031...|  0.0|
|[-0.2620979654489...|  0.0|
|[-0.9474766912369...|  0.0|
|[-0.9474766912369...|  0.0|
+--------------------+-----+
only showing top 5 rows


rename the df colms to feature and label as models needs them by default

In [44]:
final_df = final_df.withColumnRenamed(
    "scaled_features",
    "features"
)

In [45]:
final_df.printSchema()

root
 |-- features: vector (nullable = true)
 |-- label: double (nullable = false)



### ML training

In [46]:
train_df, test_df = final_df.randomSplit([0.8, 0.2], seed=42)

print("Training Records :", train_df.count())
print("Testing Records  :", test_df.count())

Training Records : 24014
Testing Records  : 6125


Logistic Reg

In [47]:
from pyspark.ml.classification import LogisticRegression, MultiClas

lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    maxIter=100,
    regParam=0.01
)

In [48]:
lr_model = lr.fit(train_df)

In [49]:
predictions = lr_model.transform(test_df)

In [59]:
predictions.select(
    "features",
    "label",
    "prediction",
    "probability"
).show(10, truncate=True)




+--------------------+-----+----------+--------------------+
|            features|label|prediction|         probability|
+--------------------+-----+----------+--------------------+
|[-1.4043958417623...|  0.0|       0.0|[0.98166099895228...|
|[-1.2520894582538...|  0.0|       0.0|[0.96615370024753...|
|[-1.2520894582538...|  0.0|       0.0|[0.99094613719778...|
|[-1.1759362664996...|  0.0|       0.0|[0.98402389485035...|
|[-1.0236298829912...|  0.0|       0.0|[0.98848094600002...|
|[-0.9474766912369...|  0.0|       0.0|[0.97121657923423...|
|[-0.8713234994827...|  0.0|       0.0|[0.98066978509340...|
|[-0.7190171159743...|  0.0|       0.0|[0.98253945477144...|
|[-0.6428639242200...|  0.0|       0.0|[0.95501389690725...|
|[-0.6428639242200...|  0.0|       0.0|[0.78679092421486...|
+--------------------+-----+----------+--------------------+
only showing top 10 rows


### confusion metrix and some other eval paras

In [56]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

for metric in ["accuracy", "weightedPrecision", "weightedRecall", "f1"]:

    evaluator = MulticlassClassificationEvaluator(
        labelCol="label",
        predictionCol="prediction",
        metricName=metric
    )

    print(f"{metric:20}: {evaluator.evaluate(predictions):.4f}")

accuracy            : 0.8526
weightedPrecision   : 0.8469
weightedRecall      : 0.8526
f1                  : 0.8463


In [57]:
predictions.groupBy(
    "label",
    "prediction"
).count().orderBy("label", "prediction").show()

+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|  0.0|       0.0| 4272|
|  0.0|       1.0|  283|
|  1.0|       0.0|  620|
|  1.0|       1.0|  950|
+-----+----------+-----+



In [60]:
print("Intercept:")
print(lr_model.intercept)

print("\nNumber of Coefficients:")
print(len(lr_model.coefficients))

print("\nCoefficients:")
print(lr_model.coefficients)

Intercept:
-1.8530202287950917

Number of Coefficients:
104

Coefficients:
[0.2829811494953632,0.0746535569491635,0.3060932954066306,1.0600774923821688,0.21988673592291913,0.31373758939981444,0.027505838166364757,-0.08720013612600738,-0.020925937959275834,-0.03798727894483098,0.057692512654932346,0.08729778179021595,-0.048584822124010454,-0.08382604594276921,-0.009940593532729022,0.14685888776907385,0.13878765171638435,0.006835522637434724,-0.0910385866478532,-0.015980776726152043,-0.1054434590289175,-0.09036258512441632,0.11911826132018989,-0.05694878871583101,-0.04291447239635177,0.09465246242606487,-0.0431364956161064,-0.04268189953875157,-0.06905756065480441,0.39298774326858016,-0.31686060953937634,-0.08067668394617324,-0.06892618398421159,-0.027551039926611006,-0.04029032630604943,0.042495667482299895,0.146143675870796,-0.016337270235633975,0.2342400369667651,-0.006798322824767979,0.061447764833588416,-0.23776421418122193,-0.06725151181964564,-0.03842095794595773,-0.13374272351257